# U9-04 · Interpretar el modelo, vigilar la deriva y análisis completo

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 9, apartados 9.8 a 9.10*

Ejecuta cada celda con **Mayús + Intro**. Para que funcione en el navegador, la flota simulada tiene **50 máquinas** en lugar de 200 (438 000 lecturas en lugar de 1 752 000): los tamaños y tiempos serán menores que en el libro y los resultados, parecidos pero no idénticos.

In [ ]:
import os, sys
if sys.platform == "emscripten":
    # En el navegador, DuckDB no lee la carpeta de JupyterLite:
    # se trabaja en una carpeta temporal en memoria.
    os.chdir("/tmp")
%pip install -q duckdb pyarrow
import duckdb, pyarrow
print("DuckDB", duckdb.__version__, "· PyArrow", pyarrow.__version__)

## 0. Los datos simulados

El generador de la unidad (archivo `lecturas.py` del libro), con 50 máquinas. Cada hora registra temperatura, vibración, consumo y horas desde la revisión; desde octubre, la mitad de la flota mide la temperatura 3 °C de más.

In [ ]:
import numpy as np
import pandas as pd

def generar_lecturas(maquinas=50, horas=24 * 365, semilla=0):
    """Lecturas horarias simuladas de una flota durante un año."""
    rng = np.random.default_rng(semilla)
    ritmo = rng.uniform(0.5, 1.5, maquinas)      # desgaste por hora
    desgaste = rng.uniform(0, 300, maquinas)     # desde la revisión
    filas = []
    for h in range(horas):
        desgaste += ritmo
        temp = 60 + 0.01 * desgaste + rng.normal(0, 2, maquinas)
        if h >= 24 * 273:              # desde octubre, media flota
            temp[: maquinas // 2] += 3  # con sensor descalibrado
        vib = 2 + 0.004 * desgaste + rng.normal(0, 0.5, maquinas)
        kw = 40 + 0.5 * (temp - 60) + rng.normal(0, 3, maquinas)
        z = -9 + 0.006 * desgaste + 0.8 * (vib - 2)
        averia = rng.random(maquinas) < 1 / (1 + np.exp(-z))
        filas.append((np.full(maquinas, h), np.arange(maquinas),
                      temp, vib, kw, desgaste.copy(), averia))
        desgaste[averia] = 0                      # se repara
    h, m, t, v, k, d, a = map(np.concatenate, zip(*filas))
    df = pd.DataFrame({
        "fecha": pd.Timestamp("2025-01-01") + pd.to_timedelta(h, "h"),
        "maquina": m, "temperatura": t, "vibracion": v,
        "consumo_kw": k, "horas_desde_revision": d, "averia": a})
    # objetivo: ¿se averiará la máquina en las próximas 24 horas?
    df = df.sort_values(["maquina", "fecha"], ignore_index=True)
    def en_24h(s):                               # ¿avería en las 24 h
        futuro = s[::-1].rolling(24, 1).max()[::-1]   # siguientes?
        return futuro.shift(-1, fill_value=0)
    proxima = df.groupby("maquina")["averia"].transform(en_24h)
    df["averia_24h"] = proxima.astype(bool)
    return df

In [ ]:
lecturas = generar_lecturas()
lecturas = lecturas.astype({"maquina": "int16", "temperatura": "float32",
    "vibracion": "float32", "consumo_kw": "float32",
    "horas_desde_revision": "float32"})
lecturas.to_parquet("lecturas.parquet", index=False)
print(len(lecturas), "lecturas guardadas en lecturas.parquet")

## 1. Importancia por permutación y dependencia parcial (ejemplo 6)

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import (permutation_importance,
                                partial_dependence)
from sklearn.metrics import roc_auc_score

X_COLS = ["temperatura", "vibracion", "consumo_kw",
          "horas_desde_revision"]
datos = pd.read_parquet("lecturas.parquet")
mes = datos["fecha"].dt.month
ent = datos[mes <= 6].sample(50_000, random_state=0)
test = datos[mes.between(7, 9)].sample(20_000, random_state=0)

def explicar(columnas):
    modelo = HistGradientBoostingClassifier(
        learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=20,
        l2_regularization=1.0, random_state=0)
    modelo.fit(ent[columnas], ent["averia_24h"])
    p = modelo.predict_proba(test[columnas])[:, 1]
    print(f"AUC = {roc_auc_score(test['averia_24h'], p):.4f}")
    imp = permutation_importance(modelo, test[columnas],
            test["averia_24h"], scoring="roc_auc", n_repeats=5,
            random_state=0)
    for col, m in zip(columnas, imp.importances_mean):
        print(f"  {col:<22} pierde {m:.4f} de AUC")
    return modelo

modelo = explicar(X_COLS)
explicar(X_COLS[:3])                 # sin horas_desde_revision

trozo = test[X_COLS].iloc[:5000].astype(float)
dp = partial_dependence(modelo, trozo, ["horas_desde_revision"],
        grid_resolution=8, method="brute",
        response_method="predict_proba")
for x, y in zip(dp["grid_values"][0], dp["average"][0]):
    print(f"{x:5.0f} h -> probabilidad media {y:.3f}")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(dp["grid_values"][0], dp["average"][0], marker="o")
plt.xlabel("Horas desde la revisión"); plt.ylabel("Probabilidad media")
plt.grid(alpha=0.3); plt.show()

## 2. Detectar la deriva (ejemplo 7)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

X_COLS = ["temperatura", "vibracion", "consumo_kw",
          "horas_desde_revision"]
datos = pd.read_parquet("lecturas.parquet")
mes = datos["fecha"].dt.month
ref = datos[mes <= 6]
ent = ref.sample(50_000, random_state=0)

def psi(esperado, actual, cortes=10):
    """Índice de estabilidad de la población."""
    bordes = np.quantile(esperado, np.linspace(0, 1, cortes + 1))
    bordes[0], bordes[-1] = -np.inf, np.inf
    e = np.histogram(esperado, bordes)[0] / len(esperado)
    a = np.histogram(actual, bordes)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

def modelo(cols):
    return HistGradientBoostingClassifier(
        learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=20,
        l2_regularization=1.0, random_state=0).fit(
            ent[cols], ent["averia_24h"])

completo, sin_horas = modelo(X_COLS), modelo(X_COLS[:3])
print("mes    KS    PSI  AUC compl.  AUC sin h.  alarmas sin h.")
for m in range(7, 13):
    lote, y = datos[mes == m], datos.loc[mes == m, "averia_24h"]
    ks = ks_2samp(ref["temperatura"], lote["temperatura"]).statistic
    p = psi(ref["temperatura"], lote["temperatura"])
    a1 = completo.predict_proba(lote[X_COLS])[:, 1]
    a2 = sin_horas.predict_proba(lote[X_COLS[:3]])[:, 1]
    auc1, auc2 = roc_auc_score(y, a1), roc_auc_score(y, a2)
    print(f"{m:>3}  {ks:.3f}  {p:.3f}    {auc1:.4f}      {auc2:.4f}"
          f"        {(a2 > 0.2).mean():.1%}")

**Prueba.** Calcula el PSI de la `vibracion` en lugar de la temperatura. ¿Detecta algún cambio?

## 3. Análisis de principio a fin (ejemplo 8)

In [ ]:
import duckdb
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# 1) variables nuevas calculadas por DuckDB sobre el Parquet
consulta = """
    SELECT *, month(fecha) AS mes,
        avg(vibracion) OVER w   AS vib_24h,
        avg(temperatura) OVER w AS temp_24h
    FROM 'lecturas.parquet'
    WINDOW w AS (PARTITION BY maquina ORDER BY fecha
                 ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)
    ORDER BY maquina, fecha
"""
datos = duckdb.sql(consulta).df()
ent = datos[datos["mes"] <= 6].sample(50_000, random_state=0)
test = datos[datos["mes"].between(7, 9)]
y = test["averia_24h"]

def avisos(puntuacion, cupo=0.05):
    """El taller revisa el 5 % de lecturas con más riesgo."""
    alarma = puntuacion >= np.quantile(puntuacion, 1 - cupo)
    return y[alarma].mean(), alarma[y].mean()  # precisión, recall

# 2) regla actual: avisar a las máquinas con más horas sin revisar
prec, rec = avisos(test["horas_desde_revision"].to_numpy())
print(f"regla de horas  precisión {prec:.2f}, "
      f"exhaustividad {rec:.2f}")

# 3) modelos con y sin las variables de ventana
grupos = {"4 lecturas": ["temperatura", "vibracion", "consumo_kw",
                         "horas_desde_revision"],
          "+ ventanas": ["temperatura", "vibracion", "consumo_kw",
                         "horas_desde_revision", "vib_24h",
                         "temp_24h"]}
for nombre, cols in grupos.items():
    modelo = HistGradientBoostingClassifier(
        learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=20,
        l2_regularization=1.0, random_state=0)
    modelo.fit(ent[cols], ent["averia_24h"])
    p = modelo.predict_proba(test[cols])[:, 1]
    prec, rec = avisos(p)
    print(f"{nombre:<11} AUC {roc_auc_score(y, p):.4f}, "
          f"precisión {prec:.2f}, exhaustividad {rec:.2f}")

**Ejercicio.** Cambia el cupo de avisos a 10 % (`cupo=0.10`). ¿Qué ganas en exhaustividad y qué pierdes en precisión?

> Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
>
> Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *The Annals of Statistics*, 29(5), 1189–1232. https://doi.org/10.1214/aos/1013203451
>
> Gama, J. et al. (2014). A survey on concept drift adaptation. *ACM Computing Surveys*, 46(4), 44. https://doi.org/10.1145/2523813